In [ ]:
import math

class TicTacToeMinimax:
    def __init__(self):
        self.N = 3
        self.board = [[' ' for _ in range(self.N)] for _ in range(self.N)]
        self.player = 'X'  # Người chơi
        self.ai = 'O'      # Máy

    def print_board(self):
        print("\n  " + "   ".join(str(i) for i in range(self.N)))
        for i in range(self.N):
            print(f"{i} " + " | ".join(self.board[i]))
            if i < self.N - 1:
                print("  " + "-" * (self.N * 4 - 3))

    def is_winner(self, player):
        # Kiểm tra hàng
        for i in range(self.N):
            if all(self.board[i][j] == player for j in range(self.N)):
                return True

        # Kiểm tra cột
        for j in range(self.N):
            if all(self.board[i][j] == player for i in range(self.N)):
                return True

        # Kiểm tra đường chéo chính
        if all(self.board[i][i] == player for i in range(self.N)):
            return True

        # Kiểm tra đường chéo phụ
        if all(self.board[i][self.N - 1 - i] == player for i in range(self.N)):
            return True

        return False

    def is_full(self):
        return all(self.board[i][j] != ' ' for i in range(self.N) for j in range(self.N))

    def is_game_over(self):
        return self.is_winner(self.player) or self.is_winner(self.ai) or self.is_full()

    def evaluate(self):
        """Đánh giá bàn cờ"""
        if self.is_winner(self.ai):
            return 10
        elif self.is_winner(self.player):
            return -10
        else:
            return 0

    def minimax(self, depth, is_maximizing):
        """
        Thuật toán Minimax
        is_maximizing = True: lượt của AI (tìm max)
        is_maximizing = False: lượt của Player (tìm min)
        """
        # Kiểm tra kết thúc game
        if self.is_winner(self.ai):
            return 10 - depth  # Ưu tiên thắng nhanh
        elif self.is_winner(self.player):
            return -10 + depth  # Ưu tiên thua chậm
        elif self.is_full():
            return 0

        if is_maximizing:
            best_score = -math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.ai
                        score = self.minimax(depth + 1, False)
                        self.board[i][j] = ' '
                        best_score = max(score, best_score)
            return best_score
        else:
            best_score = math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.player
                        score = self.minimax(depth + 1, True)
                        self.board[i][j] = ' '
                        best_score = min(score, best_score)
            return best_score

    def best_move(self):
        best_score = -math.inf
        best_move = None
        for i in range(self.N):
            for j in range(self.N):
                if self.board[i][j] == ' ':
                    self.board[i][j] = self.ai
                    score = self.minimax(0, False)
                    self.board[i][j] = ' '
                    if score > best_score:
                        best_score = score
                        best_move = (i, j)
        return best_move

    def play(self):
        print("=== TIC TAC TOE - MINIMAX ===")
        print("Bạn là X, AI là O")
        print("Nhập tọa độ theo dạng: hàng cột")

        while not self.is_game_over():
            self.print_board()

            # Lượt người chơi
            while True:
                try:
                    row, col = map(int, input("\nLượt của bạn (hàng cột): ").split())
                    if 0 <= row < self.N and 0 <= col < self.N and self.board[row][col] == ' ':
                        self.board[row][col] = self.player
                        break
                    else:
                        print("Ô không hợp lệ! Vui lòng chọn ô trống trong bảng.")
                except:
                    print("Định dạng không đúng! Nhập theo dạng: hàng cột")

            if self.is_game_over():
                break

            # Lượt AI
            print("\nAI đang suy nghĩ...")
            move = self.best_move()
            if move:
                row, col = move
                self.board[row][col] = self.ai
                print(f"AI chọn ô ({row}, {col})")

        self.print_board()
        if self.is_winner(self.ai):
            print("\n🔴 AI đã thắng!")
        elif self.is_winner(self.player):
            print("\n🟢 Bạn đã thắng! Xuất sắc!")
        else:
            print("\n⚪ Hòa!")

if __name__ == "__main__":
    game = TicTacToeMinimax()
    game.play()

In [ ]:
import math

class TicTacToeAlphaBeta:
    def __init__(self):
        self.N = 3
        self.board = [[' ' for _ in range(self.N)] for _ in range(self.N)]
        self.player = 'X'
        self.ai = 'O'
        self.nodes_pruned = 0  # Đếm số node bị cắt tỉa

    def print_board(self):
        print("\n  " + "   ".join(str(i) for i in range(self.N)))
        for i in range(self.N):
            print(f"{i} " + " | ".join(self.board[i]))
            if i < self.N - 1:
                print("  " + "-" * (self.N * 4 - 3))

    def is_winner(self, player):
        for i in range(self.N):
            if all(self.board[i][j] == player for j in range(self.N)):
                return True
        for j in range(self.N):
            if all(self.board[i][j] == player for i in range(self.N)):
                return True
        if all(self.board[i][i] == player for i in range(self.N)):
            return True
        if all(self.board[i][self.N - 1 - i] == player for i in range(self.N)):
            return True
        return False

    def is_full(self):
        return all(self.board[i][j] != ' ' for i in range(self.N) for j in range(self.N))

    def is_game_over(self):
        return self.is_winner(self.player) or self.is_winner(self.ai) or self.is_full()

    def evaluate(self):
        if self.is_winner(self.ai):
            return 10
        elif self.is_winner(self.player):
            return -10
        return 0

    def alpha_beta(self, depth, alpha, beta, is_maximizing):
        """
        Thuật toán Alpha-Beta Pruning
        alpha: giá trị tốt nhất mà người chơi MAX có thể đảm bảo
        beta: giá trị tốt nhất mà người chơi MIN có thể đảm bảo
        """
        if self.is_winner(self.ai):
            return 10 - depth
        elif self.is_winner(self.player):
            return -10 + depth
        elif self.is_full():
            return 0

        if is_maximizing:
            max_eval = -math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.ai
                        eval = self.alpha_beta(depth + 1, alpha, beta, False)
                        self.board[i][j] = ' '
                        max_eval = max(max_eval, eval)
                        alpha = max(alpha, eval)
                        if beta <= alpha:
                            self.nodes_pruned += 1
                            break
                if beta <= alpha:
                    break
            return max_eval
        else:
            min_eval = math.inf
            for i in range(self.N):
                for j in range(self.N):
                    if self.board[i][j] == ' ':
                        self.board[i][j] = self.player
                        eval = self.alpha_beta(depth + 1, alpha, beta, True)
                        self.board[i][j] = ' '
                        min_eval = min(min_eval, eval)
                        beta = min(beta, eval)
                        if beta <= alpha:
                            self.nodes_pruned += 1
                            break
                if beta <= alpha:
                    break
            return min_eval

    def best_move(self):
        best_score = -math.inf
        best_move = None
        alpha = -math.inf
        beta = math.inf
        self.nodes_pruned = 0

        for i in range(self.N):
            for j in range(self.N):
                if self.board[i][j] == ' ':
                    self.board[i][j] = self.ai
                    score = self.alpha_beta(0, alpha, beta, False)
                    self.board[i][j] = ' '
                    if score > best_score:
                        best_score = score
                        best_move = (i, j)
                    alpha = max(alpha, score)

        print(f"📊 Số node bị cắt tỉa: {self.nodes_pruned}")
        return best_move

    def play(self):
        print("=== TIC TAC TOE - ALPHA-BETA PRUNING ===")
        print("Bạn là X, AI là O (sử dụng Alpha-Beta)")

        while not self.is_game_over():
            self.print_board()

            # Lượt người chơi
            while True:
                try:
                    row, col = map(int, input("\nLượt của bạn (hàng cột): ").split())
                    if 0 <= row < self.N and 0 <= col < self.N and self.board[row][col] == ' ':
                        self.board[row][col] = self.player
                        break
                    else:
                        print("Ô không hợp lệ!")
                except:
                    print("Nhập đúng định dạng: hàng cột")

            if self.is_game_over():
                break

            print("\nAI đang suy nghĩ với Alpha-Beta...")
            move = self.best_move()
            if move:
                row, col = move
                self.board[row][col] = self.ai
                print(f"AI chọn ô ({row}, {col})")

        self.print_board()
        if self.is_winner(self.ai):
            print("\n🔴 AI thắng!")
        elif self.is_winner(self.player):
            print("\n🟢 Bạn thắng!")
        else:
            print("\n⚪ Hòa!")

if __name__ == "__main__":
    game = TicTacToeAlphaBeta()
    game.play()